# Flight Analytics — Gold Layer

Reads Silver as a stream and produces three separate Gold tables, each backing a different dashboard panel:

1. **`flight_kpis`** — windowed aggregates (active flights, avg speed/altitude, climbing/descending counts)
2. **`streaming_health`** — micro-batch throughput metrics (rows/sec, batch duration)
3. **`longest_flights`** — Haversine distance from a reference point, windowed top-N per aircraft

All three use the same event-time column and watermark as Silver, for consistency.


In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

print("Spark version:", spark.version)


In [ ]:
BASE_PATH = "/Volumes/flight_analytics/streaming/flight_data"

silver_path = f"{BASE_PATH}/silver"

kpis_path = f"{BASE_PATH}/gold/flight_kpis"
kpis_checkpoint = f"{BASE_PATH}/checkpoints/gold_flight_kpis"

health_path = f"{BASE_PATH}/gold/streaming_health"
health_checkpoint = f"{BASE_PATH}/checkpoints/gold_streaming_health"

longest_path = f"{BASE_PATH}/gold/longest_flights"
longest_checkpoint = f"{BASE_PATH}/checkpoints/gold_longest_flights"

WATERMARK_DELAY = "30 seconds"
WINDOW_DURATION = "1 minute"
WINDOW_SLIDE = "30 seconds"

print("Gold paths configured.")


In [ ]:
silver_stream = (
    spark.readStream.format("delta").load(silver_path)
    .withWatermark("time_position_ts", WATERMARK_DELAY)
)

print("Silver stream ready. Columns:", silver_stream.columns)


## 1. `flight_kpis` — windowed aggregates

Sliding 1-minute windows, refreshed every 30s, grouped by window only (not also by country/aircraft, to keep this table small and dashboard-friendly — origin-country breakdowns can be a separate cut if needed later).

Climb/descent classification uses `vertical_rate` directly from OpenSky, with a small dead-band (±1 m/s) treated as "level" rather than noise being read as a climb/descent.


In [ ]:
flight_kpis_stream = (
    silver_stream
    .groupBy(F.window("time_position_ts", WINDOW_DURATION, WINDOW_SLIDE))
    .agg(
        F.approx_count_distinct("icao24").alias("active_flight_count"),
        F.avg("velocity").alias("avg_velocity_ms"),
        F.avg("baro_altitude").alias("avg_altitude_m"),
        F.sum(F.when(F.col("vertical_rate") > 1, 1).otherwise(0)).alias("climbing_count"),
        F.sum(F.when(F.col("vertical_rate") < -1, 1).otherwise(0)).alias("descending_count"),
        F.sum(
            F.when(F.col("vertical_rate").between(-1, 1), 1).otherwise(0)
        ).alias("level_count"),
        F.avg("dq_score").alias("avg_dq_score"),
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "active_flight_count", "avg_velocity_ms", "avg_altitude_m",
        "climbing_count", "descending_count", "level_count", "avg_dq_score",
    )
)

print("flight_kpis aggregation defined.")


In [ ]:
def write_flight_kpis(batch_df, batch_id):
    if batch_df.isEmpty():
        print(f"Batch {batch_id}: no KPI rows, skipping.")
        return

    if DeltaTable.isDeltaTable(spark, kpis_path):
        tgt = DeltaTable.forPath(spark, kpis_path)
        (
            tgt.alias("tgt")
            .merge(
                batch_df.alias("src"),
                "tgt.window_start = src.window_start AND tgt.window_end = src.window_end",
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        batch_df.write.format("delta").mode("overwrite").save(kpis_path)

    print(f"Batch {batch_id}: wrote {batch_df.count()} KPI window rows.")


kpis_query = (
    flight_kpis_stream.writeStream
    .foreachBatch(write_flight_kpis)
    .option("checkpointLocation", kpis_checkpoint)
    .trigger(availableNow=True)
    .start()
)

kpis_query.awaitTermination()
print("flight_kpis write finished.")


## 2. `streaming_health` — micro-batch throughput metrics

**Note on scope:** true Kafka consumer lag (offset-based) isn't straightforward to surface reliably over the Event Hubs Kafka-compatible endpoint from inside `foreachBatch`. What *is* directly available and reliable is Spark's own per-batch progress metrics — rows processed, processing duration, and derived throughput — which is what's captured here. This is a fair proxy for pipeline health (a growing batch duration or falling throughput is the same practical signal as growing lag) and is called out honestly as a proxy, not true Kafka lag, in the project documentation.


In [ ]:
import time

def write_streaming_health(batch_df, batch_id):
    start = time.time()
    row_count = batch_df.count()
    duration_s = max(time.time() - start, 0.001)  # avoid divide-by-zero on empty/instant batches
    rows_per_sec = row_count / duration_s

    health_row = spark.createDataFrame(
        [(batch_id, row_count, duration_s, rows_per_sec)],
        ["batch_id", "row_count", "batch_duration_s", "rows_per_sec"],
    ).withColumn("recorded_at", F.current_timestamp())

    health_row.write.format("delta").mode("append").save(health_path)
    print(f"Batch {batch_id}: {row_count} rows in {duration_s:.2f}s ({rows_per_sec:.1f} rows/sec)")


health_query = (
    silver_stream.writeStream
    .foreachBatch(write_streaming_health)
    .option("checkpointLocation", health_checkpoint)
    .trigger(availableNow=True)
    .start()
)

health_query.awaitTermination()
print("streaming_health write finished.")


## 3. `longest_flights` — Haversine distance from a reference point

Computes great-circle distance (km) from a fixed reference location for every record, using the standard Haversine formula expressed directly in Spark SQL functions (no Python UDF, so it runs at native Spark speed instead of row-by-row in the JVM↔Python bridge). Windowed to find, per 1-minute window, the top 10 aircraft currently furthest from the reference point.

Reference point defaults to Bengaluru — change `REF_LAT`/`REF_LON` to any point relevant to your bounding box.


In [ ]:
REF_LAT = 12.9716
REF_LON = 77.5946
EARTH_RADIUS_KM = 6371.0

def haversine_km_expr(lat_col, lon_col, ref_lat, ref_lon):
    """Great-circle distance (km) from (ref_lat, ref_lon) to (lat_col, lon_col), as a native Spark expression."""
    lat1 = F.radians(F.lit(ref_lat))
    lat2 = F.radians(F.col(lat_col))
    dlat = F.radians(F.col(lat_col) - F.lit(ref_lat))
    dlon = F.radians(F.col(lon_col) - F.lit(ref_lon))

    a = (
        F.sin(dlat / 2) ** 2
        + F.cos(lat1) * F.cos(lat2) * F.sin(dlon / 2) ** 2
    )
    c = 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a))
    return F.lit(EARTH_RADIUS_KM) * c


distance_stream = (
    silver_stream
    .filter(F.col("latitude").isNotNull() & F.col("longitude").isNotNull())
    .withColumn(
        "distance_from_ref_km",
        haversine_km_expr("latitude", "longitude", REF_LAT, REF_LON),
    )
)

longest_flights_stream = (
    distance_stream
    .groupBy(
        F.window("time_position_ts", WINDOW_DURATION, WINDOW_SLIDE),
        "icao24",
    )
    .agg(
        F.max("distance_from_ref_km").alias("max_distance_from_ref_km"),
        F.first("callsign", ignorenulls=True).alias("callsign"),
        F.first("origin_country", ignorenulls=True).alias("origin_country"),
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "icao24", "callsign", "origin_country", "max_distance_from_ref_km",
    )
)

print("longest_flights aggregation defined.")


In [ ]:
def write_longest_flights(batch_df, batch_id):
    if batch_df.isEmpty():
        print(f"Batch {batch_id}: no rows, skipping.")
        return

    # Keep only the top 10 per window here in the batch writer - ranking is cheap on an
    # already-small per-window aggregate, and keeps the Gold table itself dashboard-sized.
    from pyspark.sql.window import Window

    rank_window = Window.partitionBy("window_start", "window_end").orderBy(
        F.col("max_distance_from_ref_km").desc()
    )
    top10_df = (
        batch_df
        .withColumn("rank", F.row_number().over(rank_window))
        .filter(F.col("rank") <= 10)
        .drop("rank")
    )

    if DeltaTable.isDeltaTable(spark, longest_path):
        tgt = DeltaTable.forPath(spark, longest_path)
        (
            tgt.alias("tgt")
            .merge(
                top10_df.alias("src"),
                "tgt.window_start = src.window_start AND tgt.window_end = src.window_end "
                "AND tgt.icao24 = src.icao24",
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        top10_df.write.format("delta").mode("overwrite").save(longest_path)

    print(f"Batch {batch_id}: wrote {top10_df.count()} longest-flight rows.")


longest_query = (
    longest_flights_stream.writeStream
    .foreachBatch(write_longest_flights)
    .option("checkpointLocation", longest_checkpoint)
    .trigger(availableNow=True)
    .start()
)

longest_query.awaitTermination()
print("longest_flights write finished.")


## Sanity checks


In [ ]:
print("--- flight_kpis ---")
display(spark.read.format("delta").load(kpis_path).orderBy(F.col("window_start").desc()).limit(10))


In [ ]:
print("--- streaming_health ---")
display(spark.read.format("delta").load(health_path).orderBy(F.col("recorded_at").desc()).limit(10))


In [ ]:
print("--- longest_flights ---")
display(
    spark.read.format("delta").load(longest_path)
    .orderBy(F.col("window_start").desc(), F.col("max_distance_from_ref_km").desc())
    .limit(20)
)
